# PolicyShift — full GPU post-training (SFT + DPO)

**Runtime → Change runtime type → GPU (T4 is fine).**

This trains real LoRA SFT + DPO on a **v1.0+v1.1 → v2.0** policy-shift split.
RL is optional and skipped here (often breaks on Colab dependency pins).

Do **not** invent metrics — download `summary.json` + checkpoints after the run.

In [ ]:
# Fresh clone (skip clone if you already have /content/policyshift)
import os
if not os.path.isdir("/content/policyshift"):
    !git clone https://github.com/ishikanahar/policyshift.git
%cd /content/policyshift
!git pull --ff-only
!pip install -q -U "torchao>=0.16.0" "peft>=0.14.0" "trl>=0.14.0"
!pip install -q -e ".[training]"
# Confirm GPU
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"

In [ ]:
# Main Cohere study: train on v1.0+v1.1, eval on v2.0, real LoRA SFT+DPO
!python scripts/run_shift_experiment.py --config configs/study/full.yaml --no-smoke

# Optional: also keep the smaller standalone checkpoints
!python scripts/prepare_full_training_data.py --n-cases 80 --out-root data/full --train-versions 1.0,1.1 --eval-versions 2.0
!python scripts/validate_no_v2_leakage.py --sft data/full/sft/sft_train.jsonl --dpo data/full/dpo/dpo_train.jsonl
!python scripts/train_sft.py --config configs/sft/full_gpu.yaml --train-file data/full/sft/sft_train.jsonl --policy-versions 1.0,1.1 --no-smoke --max-steps 50
!python scripts/train_dpo.py --config configs/dpo/full_gpu.yaml --train-file data/full/dpo/dpo_train.jsonl --policy-versions 1.0,1.1 --no-smoke
# RL intentionally omitted on Colab (torchao/peft pin fights). Not required for the standout study.

## Download these when done

Left sidebar → folder → download:

1. `artifacts/experiments/shift-study/summary.json`
2. `artifacts/experiments/shift-study/sft/checkpoints/` (folder)
3. `artifacts/experiments/shift-study/dpo/checkpoints/` (folder)
4. `artifacts/experiments/shift-study/leakage_report.json`

Then message Cursor that training finished (or paste the last JSON printout).

**Ignore RL** for this study. If an older cell tried `train_rl.py` and failed on `torchao`, that is OK — SFT+DPO are what matter.